# 01 — Data Loading & Parsing
**Source :** Freddie Mac Single-Family Loan-Level Dataset — Release 47 (sample, 1999–2025)

**Structure des fichiers :**
- `origination_sample_file.txt` → Origination (1 ligne par prêt, 31 colonnes)
- `performance_sample_file.txt` → Performance (1 ligne par prêt par mois, 35 colonnes)
- Séparateur : `|` — pas de header — encodage latin-1

**Téléchargement :** https://freddiemac.com/research/datasets/sf-loanlevel-dataset (compte Clarity requis)

In [1]:
import os, glob

# Chemin vers les fichiers bruts (relatif au notebook)
RAW_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "data", "raw")

txt_files = glob.glob(os.path.join(RAW_DIR, "*.txt"))

# Dict {nom_fichier: chemin_complet}
uploaded = {os.path.basename(p): p for p in txt_files}

if not uploaded:
    raise FileNotFoundError(
        f"Aucun fichier .txt trouvé dans {RAW_DIR}\n"
        "Place origination_sample_file.txt et performance_sample_file.txt dans data/raw/"
    )

print("Fichiers détectés :")
for name in uploaded:
    print(f"  {name}")

Fichiers détectés :
  sample_svcg_2018.txt
  sample_orig_2017.txt
  sample_orig_2018.txt
  sample_svcg_2017.txt


In [9]:
pd.set_option('display.max_columns', None)   # toutes les colonnes
pd.set_option('display.width', None)         # pas de wrap horizontal
pd.set_option('display.max_colwidth', 30)    # tronque chaque cellule à 30 caractères

In [2]:
import pandas as pd
import numpy as np

# Colonnes Origination — 32 colonnes, Standard Dataset actuel
ORIG_COLS = [
    'credit_score',                    # 1
    'first_payment_date',              # 2
    'first_time_homebuyer_flag',       # 3
    'maturity_date',                   # 4
    'msa',                             # 5
    'mip',                             # 6
    'units',                           # 7
    'occupancy_status',                # 8
    'ocltv',                           # 9
    'dti',                             # 10
    'original_upb',                    # 11
    'oltv',                            # 12
    'original_interest_rate',          # 13
    'channel',                         # 14
    'ppm_flag',                        # 15
    'amortization_type',               # 16
    'property_state',                  # 17
    'property_type',                   # 18
    'postal_code',                     # 19
    'loan_sequence_number',            # 20
    'loan_purpose',                    # 21
    'original_loan_term',              # 22
    'number_of_borrowers',             # 23
    'seller_name',                     # 24
    'servicer_name',                   # 25
    'super_conforming_flag',           # 26
    'pre_harp_loan_sequence_number',   # 27
    'program_indicator',               # 28
    'harp_indicator',                  # 29
    'property_valuation_method',       # 30
    'interest_only_indicator',         # 31
    'mi_cancellation_indicator',       # 32
]

# Colonnes Performance mensuelle — 32 colonnes, Standard Dataset actuel
PERF_COLS = [
    'loan_sequence_number',                # 1
    'monthly_reporting_period',            # 2
    'current_actual_upb',                  # 3
    'current_loan_delinquency_status',     # 4
    'loan_age',                            # 5
    'remaining_months_to_legal_maturity',  # 6
    'defect_settlement_date',              # 7
    'modification_flag',                   # 8
    'zero_balance_code',                   # 9
    'zero_balance_effective_date',         # 10
    'current_interest_rate',               # 11
    'current_deferred_upb',                # 12
    'ddlpi',                               # 13
    'mi_recoveries',                       # 14
    'net_sales_proceeds',                  # 15
    'non_mi_recoveries',                   # 16
    'expenses',                            # 17
    'legal_costs',                         # 18
    'maintenance_and_preservation_costs',  # 19
    'taxes_and_insurance',                 # 20
    'miscellaneous_expenses',              # 21
    'actual_loss_calculation',             # 22
    'modification_cost',                   # 23
    'step_modification_flag',              # 24
    'deferred_payment_plan',               # 25
    'estimated_ltv',                       # 26
    'zero_balance_removal_upb',            # 27
    'delinquent_accrued_interest',         # 28
    'delinquency_due_to_disaster',         # 29
    'borrower_assistance_status_code',     # 30
    'current_month_modification_cost',     # 31
    'interest_bearing_upb',                # 32
]

In [3]:
def load_origination(filepath: str) -> pd.DataFrame:
    df = pd.read_csv(
        filepath, sep='|', header=None,
        names=ORIG_COLS, dtype=str,
        encoding='latin-1', low_memory=False
    )
    assert df.shape[1] == len(ORIG_COLS), f"Origination : {df.shape[1]} colonnes lues, {len(ORIG_COLS)} attendues"
    return df

def load_performance(filepath: str) -> pd.DataFrame:
    df = pd.read_csv(
        filepath, sep='|', header=None,
        names=PERF_COLS, dtype=str,
        encoding='latin-1', low_memory=False
    )
    assert df.shape[1] == len(PERF_COLS), f"Performance : {df.shape[1]} colonnes lues, {len(PERF_COLS)} attendues"
    return df

orig_frames, perf_frames = [], []

# sample_orig_YYYY.txt → origination | sample_svcg_YYYY.txt → performance
for fname, fpath in uploaded.items():
    if 'orig' in fname.lower():
        orig_frames.append(load_origination(fpath))
    elif 'svcg' in fname.lower() or 'time' in fname.lower():
        perf_frames.append(load_performance(fpath))

assert orig_frames, "Aucun fichier origination détecté (nom doit contenir 'orig')"
assert perf_frames, "Aucun fichier performance détecté (nom doit contenir 'svcg')"

df_orig = pd.concat(orig_frames, ignore_index=True)
df_perf = pd.concat(perf_frames, ignore_index=True)

print(f"Origination : {df_orig.shape[0]:,} prêts × {df_orig.shape[1]} colonnes")
print(f"Performance : {df_perf.shape[0]:,} observations × {df_perf.shape[1]} colonnes")
print(f"Ratio moyen : {df_perf.shape[0] / df_orig.shape[0]:.1f} mois/prêt")

Origination : 100,000 prêts × 32 colonnes
Performance : 4,723,094 observations × 32 colonnes
Ratio moyen : 47.2 mois/prêt


In [4]:
# Nettoyage : types numériques + codes spéciaux Freddie Mac
NUMERIC_ORIG = [
    'credit_score', 'dti', 'original_upb', 'oltv', 'ocltv',
    'original_interest_rate', 'original_loan_term', 'mip', 'units'
]

for col in NUMERIC_ORIG:
    df_orig[col] = pd.to_numeric(df_orig[col], errors='coerce')

# Codes sentinel → NaN (source : Freddie Mac data dictionary)
df_orig.loc[df_orig['dti'] == 999, 'dti'] = np.nan
df_orig.loc[df_orig['oltv'] == 999, 'oltv'] = np.nan
df_orig.loc[df_orig['ocltv'] == 999, 'ocltv'] = np.nan
df_orig.loc[df_orig['credit_score'].isin([9999, 999]), 'credit_score'] = np.nan

print("Valeurs manquantes origination (colonnes numériques) :")
missing = df_orig[NUMERIC_ORIG].isnull().mean().sort_values(ascending=False)
print(missing[missing > 0].map('{:.1%}'.format))

Valeurs manquantes origination (colonnes numériques) :
dti             2.3%
credit_score    0.0%
oltv            0.0%
ocltv           0.0%
dtype: object


In [5]:
# Feature engineering : agrégation des données de performance → 1 ligne par prêt
df_perf['current_loan_delinquency_status'] = pd.to_numeric(
    df_perf['current_loan_delinquency_status'].replace({'X': np.nan, 'XX': np.nan, 'RA': np.nan}),
    errors='coerce'
)
df_perf['loan_age'] = pd.to_numeric(df_perf['loan_age'], errors='coerce')

perf_agg = df_perf.groupby('loan_sequence_number').agg(
    max_delinquency   = ('current_loan_delinquency_status', 'max'),
    months_delinquent = ('current_loan_delinquency_status', lambda x: (x > 0).sum()),
    ever_modified     = ('modification_flag', lambda x: (x == 'Y').any()),
    zero_balance_code = ('zero_balance_code', 'last'),
    loan_age_months   = ('loan_age', 'max'),
).reset_index()

# Cible : défaut = 90+ jours de retard OU foreclosure / REO
# zero_balance_code : '03' = foreclosure, '09' = REO
DEFAULT_ZERO_BALANCE_CODES = {'03', '09'}
perf_agg['default'] = (
    (perf_agg['max_delinquency'] >= 3) |
    (perf_agg['zero_balance_code'].isin(DEFAULT_ZERO_BALANCE_CODES))
).astype(int)

print(f"Taux de défaut observé : {perf_agg['default'].mean():.2%}")
print(perf_agg['default'].value_counts().rename({0: 'Pas de défaut', 1: 'Défaut'}))

Taux de défaut observé : 5.57%
default
Pas de défaut    94428
Défaut            5572
Name: count, dtype: int64


In [10]:
# Diagnostic + jointure sur loan_sequence_number
df_orig['loan_sequence_number'] = df_orig['loan_sequence_number'].str.strip()
perf_agg['loan_sequence_number'] = perf_agg['loan_sequence_number'].str.strip()

orig_ids = set(df_orig['loan_sequence_number'].dropna())
perf_ids = set(perf_agg['loan_sequence_number'].dropna())
overlap = orig_ids & perf_ids
print(f"IDs uniques origination : {len(orig_ids)}")
print(f"IDs uniques performance : {len(perf_ids)}")
print(f"IDs en commun          : {len(overlap)}")

df = df_orig.merge(perf_agg, on='loan_sequence_number', how='inner')
print(f"\nDataset final : {df.shape[0]:,} prêts × {df.shape[1]} variables")
print(f"\nTop 10 variables avec valeurs manquantes :")
print(df.isnull().mean().sort_values(ascending=False).head(10).map('{:.1%}'.format))
df.head(10)

IDs uniques origination : 100000
IDs uniques performance : 100000
IDs en commun          : 100000

Dataset final : 100,000 prêts × 38 variables

Top 10 variables avec valeurs manquantes :
pre_harp_loan_sequence_number    97.7%
harp_indicator                   97.7%
super_conforming_flag            96.6%
zero_balance_code                23.1%
msa                              10.6%
dti                               2.3%
credit_score                      0.0%
ocltv                             0.0%
oltv                              0.0%
property_valuation_method         0.0%
dtype: object


,credit_score,first_payment_date,first_time_homebuyer_flag,maturity_date,msa,mip,units,occupancy_status,ocltv,dti,original_upb,oltv,original_interest_rate,channel,ppm_flag,amortization_type,property_state,property_type,postal_code,loan_sequence_number,loan_purpose,original_loan_term,number_of_borrowers,seller_name,servicer_name,super_conforming_flag,pre_harp_loan_sequence_number,program_indicator,harp_indicator,property_valuation_method,interest_only_indicator,mi_cancellation_indicator,max_delinquency,months_delinquent,ever_modified,zero_balance_code,loan_age_months,default
0,809.0,201705,N,204704,NaN,0,1,P,75.0,38.0,195000,75.0,4.250,R,N,FRM,PA,SF,17900,F17Q10000002,N,360,01,Other sellers,SPECIALIZED LOAN SERVICING...,NaN,NaN,9,NaN,7,N,7,0.0,0,False,01,56,0
1,702.0,201703,N,203202,NaN,0,1,P,80.0,36.0,187000,76.0,3.000,R,N,FRM,KY,SF,42000,F17Q10000017,N,180,02,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,7,0.0,0,False,None,103,0
2,792.0,201703,N,204702,NaN,0,1,S,60.0,36.0,87000,60.0,3.375,R,N,FRM,MI,SF,48700,F17Q10000064,N,360,02,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,7,0.0,0,False,01,60,0
3,776.0,201703,N,204702,NaN,0,1,S,80.0,18.0,106000,80.0,4.250,R,N,FRM,NY,SF,12700,F17Q10000065,P,360,02,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,7,0.0,0,False,01,41,0
4,790.0,201703,N,204702,41620,0,1,I,75.0,42.0,75000,75.0,4.250,C,N,FRM,UT,CO,84100,F17Q10000073,C,360,01,Other sellers,"PNC BANK, NA",NaN,NaN,9,NaN,7,N,7,1.0,1,False,None,103,0
5,726.0,201703,N,204702,39820,0,1,P,80.0,22.0,245000,80.0,3.875,R,N,FRM,CA,SF,96000,F17Q10000094,C,360,02,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,7,0.0,0,False,None,103,0
6,687.0,201703,N,204702,46520,0,1,P,80.0,45.0,619000,75.0,3.875,R,N,FRM,HI,PU,96700,F17Q10000176,C,360,02,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,7,8.0,12,False,01,56,1
7,766.0,201703,N,204702,NaN,0,1,P,72.0,27.0,185000,72.0,3.875,R,N,FRM,CO,SF,81200,F17Q10000300,C,360,02,Other sellers,U.S. BANK N.A.,NaN,NaN,9,NaN,7,N,7,0.0,0,False,01,32,0
8,794.0,201703,N,204702,46520,0,1,P,60.0,19.0,258000,60.0,3.375,R,N,FRM,HI,CO,96800,F17Q10000330,C,360,01,Other sellers,U.S. BANK N.A.,NaN,NaN,9,NaN,7,N,7,0.0,0,False,None,103,0
9,804.0,201703,N,204702,NaN,0,1,P,80.0,32.0,132000,80.0,3.500,R,N,FRM,KS,SF,67600,F17Q10000408,P,360,02,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,7,0.0,0,False,01,38,0


In [7]:
import os
out_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "data", "freddie_mac_clean.parquet")
df.to_parquet(out_path, index=False)
print(f"Sauvegardé : {out_path}")
print(f"Taille : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB en mémoire")

Sauvegardé : /Users/ibrahimawane/Desktop/data project/notebooks/../data/freddie_mac_clean.parquet
Taille : 132.2 MB en mémoire
